In [173]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, DecimalType, FloatType

In [174]:
spark = (
    SparkSession.builder
        .appName("minio_app_ minio")
        .master("spark://spark-master:7077")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.access.key", "admin")
        .config("spark.hadoop.fs.s3a.secret.key", "admin123")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")

        .getOrCreate()
)

In [175]:

bronze_path = "s3a://analytics/medallion/01-bronze/vendas"

In [176]:
df_silver = spark.read.format("delta").load(bronze_path)

In [177]:
df_silver.printSchema()

root
 |-- Data_da_Venda: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- PrecoUnitario: double (nullable = true)
 |-- Custo_Unitário: double (nullable = true)
 |-- Marca: string (nullable = true)
 |-- Qtd_Vendida: double (nullable = true)
 |-- Nome_Cliente: string (nullable = true)
 |-- Localidade: string (nullable = true)



In [178]:
df_silver.show(5)

+-------------------+--------------------+--------------+-------------+--------------+--------+-----------+-----------------+--------------------+
|      Data_da_Venda|             Produto|     Categoria|PrecoUnitario|Custo_Unitário|   Marca|Qtd_Vendida|     Nome_Cliente|          Localidade|
+-------------------+--------------------+--------------+-------------+--------------+--------+-----------+-----------------+--------------------+
|2017-06-01 00:00:00|Sistema de Som 7....|Sistema de Som|       1109.0|        367.43|Litware |        1.0|Pinheiro, Vicente|     França - Europa|
|2017-06-01 00:00:00|180 CFM Ventilado...|  Ventiladores|       215.62|         71.44| Litware|        1.0|    Lopez, Marlon|Estados Unidos - ...|
|2017-06-01 00:00:00|180 CFM Ventilado...|  Ventiladores|       215.62|         71.44| Litware|        1.0| Gonzalez, Martin|Chile - América d...|
|2017-06-01 00:00:00|180 CFM Ventilado...|  Ventiladores|       215.62|         71.44| Litware|        1.0|     Ruiz, 

In [179]:
df_silver.count()

203888

In [180]:
# Contar valores nulos por coluna no DataFrame
for coluna in df_silver.columns:
    nulos = df_silver.filter(F.col(coluna).isNull()).count()
    print(f"{coluna}: {nulos}")


Data_da_Venda: 0
Produto: 6
Categoria: 6
PrecoUnitario: 6
Custo_Unitário: 6
Marca: 6
Qtd_Vendida: 6
Nome_Cliente: 6
Localidade: 6


In [181]:
df_silver.filter(F.col("Produto").isNull()).show()

+-------------+-------+---------+-------------+--------------+-----+-----------+------------+----------+
|Data_da_Venda|Produto|Categoria|PrecoUnitario|Custo_Unitário|Marca|Qtd_Vendida|Nome_Cliente|Localidade|
+-------------+-------+---------+-------------+--------------+-----+-----------+------------+----------+
|             |   NULL|     NULL|         NULL|          NULL| NULL|       NULL|        NULL|      NULL|
|             |   NULL|     NULL|         NULL|          NULL| NULL|       NULL|        NULL|      NULL|
|             |   NULL|     NULL|         NULL|          NULL| NULL|       NULL|        NULL|      NULL|
|             |   NULL|     NULL|         NULL|          NULL| NULL|       NULL|        NULL|      NULL|
|             |   NULL|     NULL|         NULL|          NULL| NULL|       NULL|        NULL|      NULL|
|             |   NULL|     NULL|         NULL|          NULL| NULL|       NULL|        NULL|      NULL|
+-------------+-------+---------+-------------+--------

In [182]:
df_silver.filter(F.col("Data_da_Venda").isNull()).show()

+-------------+-------+---------+-------------+--------------+-----+-----------+------------+----------+
|Data_da_Venda|Produto|Categoria|PrecoUnitario|Custo_Unitário|Marca|Qtd_Vendida|Nome_Cliente|Localidade|
+-------------+-------+---------+-------------+--------------+-----+-----------+------------+----------+
+-------------+-------+---------+-------------+--------------+-----+-----------+------------+----------+



In [183]:
# substituir strings vazias por NULL
df_silver = df_silver.withColumn(
    "Data_da_Venda",
    F.when(F.trim(F.col("Data_da_Venda")) == "", None).otherwise(F.col("Data_da_Venda"))
)

In [184]:
# Remover linhas onde Data_da_Venda está vazia ou nula
df_silver = df_silver.filter(F.col("Data_da_venda").isNotNull())

In [185]:
df_silver = df_silver.withColumn(
    "Total_Venda", F.col("Qtd_Vendida") * F.col("PrecoUnitario")
)

In [186]:
# Criar colunas de nome e sobrenome a partir de Nome_Cliente e juntar em cliente
df_silver = df_silver.withColumn(
    "nome",
    F.split(F.col("Nome_Cliente"), ",")[1]
).withColumn(
    "sobrenome",
    F.split(F.col("Nome_Cliente"), ",")[0]
).withColumn(
    "cliente",
    F.concat(F.col("nome"), F.lit(" "), F.col("sobrenome"))
)

In [187]:
# Remover colunas temporárias após criar a coluna 'cliente'
colunas = ["nome", "sobrenome", "Nome_Cliente"]

df_silver = df_silver.drop(*colunas)

In [188]:
# Separar Localidade em Pais e Continente e remover a coluna original
df_silver = (
    df_silver.withColumn("Pais", F.trim(F.split(F.col("Localidade"), "-")[0]))
             .withColumn("Continente", F.trim(F.split(F.col("Localidade"), "-")[1]))
             .drop("Localidade")
)


In [189]:
# Remove espaços em branco extras das colunas de texto para padronizar os dados
df_silver = (
    df_silver.withColumn("Produto", F.trim(F.col("Produto"))) 
             .withColumn("Categoria", F.trim(F.col("Categoria"))) 
             .withColumn("Marca", F.trim(F.col("Marca"))) 
             .withColumn("cliente", F.trim(F.col("cliente")))
             .withColumn("Pais", F.trim(F.col("Pais")))
             .withColumn("Continente", F.trim(F.col("Continente")))
)


In [190]:
df_silver.select("Marca").distinct().show(10)

+--------------------+
|               Marca|
+--------------------+
|             Litware|
|             Contoso|
|        Hashtag Toys|
|    Southridge Video|
|           Proseware|
|Wide World Importers|
|   Northwind Traders|
+--------------------+



In [191]:
# Padronizar nomes de colunas para formato snake_case
df_silver = (
    df_silver.withColumnRenamed("Data_da_Venda", "data_venda")
             .withColumnRenamed("Produto", "produto")
             .withColumnRenamed("Categoria", "categoria")
             .withColumnRenamed("PrecoUnitario", "preco_unitario")
             .withColumnRenamed("Custo_Unitário", "custo_unitario")
             .withColumnRenamed("Marca", "marca")
             .withColumnRenamed("Qtd_Vendida", "quantidade")
             .withColumnRenamed("cliente", "cliente_nome")
             .withColumnRenamed("Continente", "continente")
             .withColumnRenamed("Total_Venda", "total_venda")
)


In [192]:
# Limpa espaços extras da coluna 'data_venda', garantindo que não haja espaços em branco no início ou no fim
df_silver =  df_silver.withColumn("data_venda", F.trim(F.col("data_venda"))) 

In [193]:
# Conta valores nulos na coluna
df_silver.filter(F.col("data_venda") == "").count()

# Mostra algumas linhas que ainda estão vazias ou nulas
df_silver.filter(F.col("data_venda") == "   ").show(5)


+----------+-------+---------+--------------+--------------+-----+----------+-----------+------------+----+----------+
|data_venda|produto|categoria|preco_unitario|custo_unitario|marca|quantidade|total_venda|cliente_nome|Pais|continente|
+----------+-------+---------+--------------+--------------+-----+----------+-----------+------------+----+----------+
+----------+-------+---------+--------------+--------------+-----+----------+-----------+------------+----+----------+



In [194]:

# Trocar vazio ou só espaços por null
df_silver = df_silver.withColumn(
    "data_venda",
    F.when(F.trim(F.col("data_venda")) == "", None).otherwise(F.col("data_venda"))
)

# Agora converte para date
df_silver = df_silver.withColumn(
    "data_venda",
    F.to_date(F.col("data_venda"), "yyyy-MM-dd HH:mm:ss")
)


In [195]:
# Converte colunas numéricas para Decimal(10,2) para padronizar precisão e escala
colunas_decimal = ["preco_unitario", "custo_unitario", "total_venda"]

for coluna in colunas_decimal:
    df_silver = df_silver.withColumn(
        coluna,
        F.col(coluna).cast(DecimalType(10, 2))
    )


In [196]:
# Converte a coluna 'quantidade' para tipo inteiro para padronização e cálculos corretos
df_silver = df_silver.withColumn("quantidade", F.col("quantidade").cast(IntegerType()))

In [197]:
df_silver.show(5)

+----------+--------------------+------------------+--------------+--------------+------------+----------+-----------+-------------+--------------+----------------+
|data_venda|             produto|         categoria|preco_unitario|custo_unitario|       marca|quantidade|total_venda| cliente_nome|          Pais|      continente|
+----------+--------------------+------------------+--------------+--------------+------------+----------+-----------+-------------+--------------+----------------+
|2017-12-24|Hand Games women ...|Jogos de Tabuleiro|          8.99|          4.13|Hashtag Toys|         3|      26.97| Pamela Vance|   Quirguistão|            Ásia|
|2017-12-24|Hand Games women ...|Jogos de Tabuleiro|          8.99|          4.13|Hashtag Toys|         4|      35.96|Ronald Kapoor|Estados Unidos|América do Norte|
|2017-12-24|Hand Games women ...|Jogos de Tabuleiro|          8.99|          4.13|Hashtag Toys|         1|       8.99|Candace Patel|        França|          Europa|
|2017-12-2

In [198]:
df_silver.printSchema()

root
 |-- data_venda: date (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- preco_unitario: decimal(10,2) (nullable = true)
 |-- custo_unitario: decimal(10,2) (nullable = true)
 |-- marca: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- total_venda: decimal(10,2) (nullable = true)
 |-- cliente_nome: string (nullable = true)
 |-- Pais: string (nullable = true)
 |-- continente: string (nullable = true)



In [199]:
df_silver = df_silver.select(
    "data_venda",
    "cliente_nome",
    "Pais",
    "continente",
    "categoria",
    "marca",
    "produto",
    "quantidade",
    "preco_unitario",
    "total_venda"
)

In [200]:
df_silver.show(5)

+----------+-------------+--------------+----------------+------------------+------------+--------------------+----------+--------------+-----------+
|data_venda| cliente_nome|          Pais|      continente|         categoria|       marca|             produto|quantidade|preco_unitario|total_venda|
+----------+-------------+--------------+----------------+------------------+------------+--------------------+----------+--------------+-----------+
|2017-12-24| Pamela Vance|   Quirguistão|            Ásia|Jogos de Tabuleiro|Hashtag Toys|Hand Games women ...|         3|          8.99|      26.97|
|2017-12-24|Ronald Kapoor|Estados Unidos|América do Norte|Jogos de Tabuleiro|Hashtag Toys|Hand Games women ...|         4|          8.99|      35.96|
|2017-12-24|Candace Patel|        França|          Europa|Jogos de Tabuleiro|Hashtag Toys|Hand Games women ...|         1|          8.99|       8.99|
|2017-12-24|Krista Martin|Estados Unidos|América do Norte|Jogos de Tabuleiro|Hashtag Toys|Hand Games

In [201]:
# Adiciona coluna com timestamp atual (data + hora)
df_silver = df_silver.withColumn("updated_at", F.current_timestamp())

In [202]:
silver_path = "s3a://analytics/medallion/02-silver/vendas"

In [203]:
# Salvando o dataframe limpo no Silver (MinIO / S3)

(
    df_silver.write 
        .format("delta")    
        .mode("overwrite") 
        .save(silver_path)
)

In [204]:
# Criar database/schema
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# Criar tabela Delta apontando para o caminho
spark.sql(""" 
CREATE TABLE IF NOT EXISTS silver.vendas
USING DELTA
LOCATION 's3a://analytics/medallion/02-silver/vendas'
""")


DataFrame[]

In [205]:
spark.sql("SELECT * FROM silver.vendas LIMIT 10 ").show() 

+----------+----------------+--------------+----------------+--------------------+------------+--------------------+----------+--------------+-----------+--------------------+
|data_venda|    cliente_nome|          Pais|      continente|           categoria|       marca|             produto|quantidade|preco_unitario|total_venda|          updated_at|
+----------+----------------+--------------+----------------+--------------------+------------+--------------------+----------+--------------+-----------+--------------------+
|2018-09-10|   Isaiah Wright|        Brasil|  América do Sul|      Sistema de Som|     Litware|Sistema de Som 5....|         1|        579.00|     579.00|2026-03-11 17:44:...|
|2018-09-10|    Sierra Young|Estados Unidos|América do Norte|      Sistema de Som|     Litware|Sistema de Som 5....|         3|        579.00|    1737.00|2026-03-11 17:44:...|
|2018-09-10|   Cameron Lewis|Estados Unidos|América do Norte|      Sistema de Som|     Litware|Sistema de Som 5....|    

In [206]:
spark.sql(" DESCRIBE DETAIL silver.vendas ").show()

+------+--------------------+--------------------+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|                name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+--------------------+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|c19e7d70-a904-4a0...|spark_catalog.sil...|       NULL|s3a://analytics/m...|2026-03-11 17:44:...|2026-03-11 17:44:42|              []|               []|       4|    1688691|        {}|               1|               2|[appendOnly, inva...|
+------+----

In [207]:
spark.sql(" DESCRIBE HISTORY silver.vendas ").show()

+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2026-03-11 17:44:42|  NULL|    NULL|    WRITE|{mode -> Overwrit...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 4, n...|        NULL|Apache-Spark/3.5....|
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+



In [208]:
df_silver = spark.read.format("delta").table("silver.vendas")
df_silver.show(5)

+----------+--------------+--------------+----------------+--------------+------------+--------------------+----------+--------------+-----------+--------------------+
|data_venda|  cliente_nome|          Pais|      continente|     categoria|       marca|             produto|quantidade|preco_unitario|total_venda|          updated_at|
+----------+--------------+--------------+----------------+--------------+------------+--------------------+----------+--------------+-----------+--------------------+
|2018-09-10| Isaiah Wright|        Brasil|  América do Sul|Sistema de Som|     Litware|Sistema de Som 5....|         1|        579.00|     579.00|2026-03-11 17:44:...|
|2018-09-10|  Sierra Young|Estados Unidos|América do Norte|Sistema de Som|     Litware|Sistema de Som 5....|         3|        579.00|    1737.00|2026-03-11 17:44:...|
|2018-09-10| Cameron Lewis|Estados Unidos|América do Norte|Sistema de Som|     Litware|Sistema de Som 5....|         2|        579.00|    1158.00|2026-03-11 17:

In [209]:
spark.stop()